# PySpark — Salting, Quick Notes

## What it is

Adding a random column (`salt`) to a skewed key so it hashes into **multiple** shuffle partitions instead of one. Doesn't reduce total data moved — it redistributes where it lands, so no single partition/core is overloaded.

<img src="https://miro.medium.com/v2/resize:fit:1100/format:webp/0*1Cy8afiMP_1vzZ86.png" height = 300/>

## Why skew happens without it

Partition assignment after a shuffle: `hash(key) % num_partitions`. Every row with the same key always hashes to the **same** partition — so one dominant key value means one overloaded partition, regardless of how many partitions exist.

## The three steps

1. **Pick a salt number.** How many pieces to split the skewed key into. A sensible default: match `spark.sql.shuffle.partitions`. Too high → tiny fragmented partitions; too low → key still concentrated.
   ```python
   SALT_NUMBER = int(spark.conf.get("spark.sql.shuffle.partitions"))
   ```
2. **Add a random salt to the skewed side** — this breaks the key into `SALT_NUMBER` hash outcomes instead of one:
   ```python
   df_skew = df_skew.withColumn("salt", (F.rand() * SALT_NUMBER).cast("int"))
   ```
3. **Join key becomes `(original_key, salt)`, never salt alone.** New partitioning rule: `hash(key, salt) % num_partitions`.

## Joins — the other side needs every salt value, not a random one

If only the skewed side gets a random salt, keys won't line up — the skewed row might get `salt=2`, and the other side has no way to know it needs `value=2, salt=2` too. So the **uniform/smaller side** gets exploded so every key appears once per possible salt value:

```python
df_uniform = (df_uniform
    .withColumn("salt_values", F.array([F.lit(i) for i in range(SALT_NUMBER)]))
    .withColumn("salt", F.explode("salt_values")))

df_joined = df_skew.join(df_uniform, ["value", "salt"], "inner")
```

**Cost trade-off:** `explode` multiplies that table's row count by `SALT_NUMBER` — expensive. Always explode the **smaller side** of the join. If both sides are similar size, there's no choice, either works.

## Aggregations — two-step group-by

Grouping by `(key, salt)` alone gives partial, fragmented counts — not the final answer. A second group-by collapses them:

```python
partial = (df_skew
    .withColumn("salt", (F.rand() * SALT_NUMBER).cast("int"))
    .groupBy("value", "salt")
    .agg(F.count("value").alias("count")))     # step 1: salted partial counts

final = (partial
    .groupBy("value")
    .agg(F.sum("count").alias("count")))       # step 2: collapse back to true totals
```

Step 1's shuffle is now spread across partitions since the key includes `salt`. Step 2's shuffle is cheap — it's aggregating small, already-reduced partial counts, not the full raw data.

## Sanity check

With very few distinct key values (e.g. only 4), post-salt partition counts can look sparse/uneven even though salting worked correctly — there just aren't many groups to spread across `SALT_NUMBER` partitions. The effect is clearest on a genuinely large skewed key (e.g. one value with ~1M rows), where it visibly spreads from one partition to several roughly-equal ones.


In [74]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark.conf.set("spark.sql.adaptive.enabled", "false")          # or just coalescePartitions.enabled = false
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # force sort-merge join
spark.conf.set("spark.sql.shuffle.partitions", "8")

## Simluate Skewed Joins

In [75]:
n = 1_000_000

df_uniform = (
    spark.createDataFrame(
        [(i,) for i in range(n)],
        schema=StructType([StructField("value", IntegerType(), False)])
    )
)

df_uniform.show(5)

In [76]:
print(f"Total Partitions that spark applied to this dataframe: {df_uniform.rdd.getNumPartitions()}")

In [77]:
(
    df_uniform
    .withColumn("partition", F.spark_partition_id())
    .groupBy('partition').count()
).show()

In [88]:
df0 = (spark.createDataFrame(
          [(0,)] * 999_990,
          schema=StructType([StructField("value", IntegerType(), False)])
      )
      .repartition(1))

df1 = (spark.createDataFrame(
          [(1,)] * 15,
          schema=StructType([StructField("value", IntegerType(), False)])
      )
      .repartition(1))

df2 = (spark.createDataFrame(
          [(2,)] * 10,
          schema=StructType([StructField("value", IntegerType(), False)])
      )
      .repartition(1))

df3 = (spark.createDataFrame(
          [(3,)] * 5,
          schema=StructType([StructField("value", IntegerType(), False)])
      )
      .repartition(1))

df_skew = (df0.union(df1).union(df2).union(df3))

In [79]:
(
    df_skew
    .withColumn('partition', F.spark_partition_id())
    .groupby('partition').count()
).show()

In [80]:
df_joined = df_skew.join(df_uniform, "value", "inner")
df_joined = (
    df_joined
    .withColumn("partition", F.spark_partition_id())
)

df_joined.groupBy('partition').count().show()

This data looks heavily skewed to the partition `0`

## Salting

In [81]:
SALT_NUMBER = int(spark.conf.get('spark.sql.shuffle.partitions'))
# SALT_NUMBER = 3
SALT_NUMBER

In [82]:
df_skew = df_skew.withColumn('salt', (F.rand() * SALT_NUMBER).cast('int'))

In [83]:
df_skew.show(10)

## Salting to uniform table

In [84]:
df_uniform = (
    df_uniform
    .withColumn('salt_values', F.array([F.lit(i) for i in range(SALT_NUMBER)]))
    .withColumn('salt', F.explode(F.col('salt_values')))
)

In [85]:
df_uniform.show(10)

## Salting in Joins

In [86]:
df_joined = df_skew.join(df_uniform, ["value", "salt"], "inner")

In [87]:
(
    df_joined
    .withColumn("partition", F.spark_partition_id())
    .groupBy('partition')
    .count()
).show()

## Salting in Aggregations

In [89]:
df_skew.groupby('value').count().show()

In [95]:
df_skew.show(5)

In [98]:
(
    df_skew
    .withColumn('partition', F.spark_partition_id())
    .groupby('partition')
    .count()
    .show()
)

In [99]:
(
    df_skew
    .withColumn('salt', (F.rand() * SALT_NUMBER).cast('int'))
    .groupBy('value')
    .agg(F.count('value').alias('count'))
    .withColumn('partition', F.spark_partition_id())
    .groupby('partition')
    .count()
    .show()
)

In [97]:
(
    df_skew
    .withColumn('salt', (F.rand() * SALT_NUMBER).cast('int'))
    .groupBy('value')
    .agg(F.count('value').alias('count'))
    .groupBy('value')
    .agg(F.sum('count').alias('count'))
    .show()
)